After TT decomposition, we have the 3 TT cores.

For example for rank (1, 15, 10, 1) and tensor (200, 30, 1401):
- G1 = 200 × 15
- G2 = 15 × 30 × 10
- G3 = 10 × 1401

These cores contain the information needed to reconstruct the MEG tensor approximately.
We need to turn them into a feature vector that represents the data.

Features

The TT cores contain the compressed information.
Construct a feature representation from them.
Conceptually:
X→(G1,G2,G3)→f

Dimensionality reduction

You cannot put 21,510 features into a small VQC.
So:
f ∈ R ^ 21510
↓
PCA
↓
z ∈ R ^ 4

Angle Encoding

z1 → Ry(z1) → q0
z2 → Ry(z2) → q1
z3 → Ry(z3) → q2
z4 → Ry(z4) → q3

Encoding
   ↓
RY/RZ trainable gates
   ↓
CNOT entanglement
   ↓
RY/RZ trainable gates
   ↓
CNOT entanglement
   ↓
Measurement

TT compression reduces redundancy while preserving an approximation of the original tensor. PCA/feature selection afterwards is a separate step whose purpose is to make the representation small enough for the quantum circuit.
PCA / feature selection: Finds a small number of directions/features that are useful for the classification problem.

(PCA) is a statistical method. It simplifies complex data by reducing the number of dimensions. It turns many correlated variables into fewer new variables called principal components. These new components keep most of the important information from the original data.

MEG feature
    ↓
VQC
    ↓
prediction
    ↓
loss
    ↓
classical optimizer
    ↓
update θ
    ↓
VQC again
    ↓
...

NPZ files
   ↓
load 4 subjects × 4 tasks
   ↓
200 trials per dataset
   ↓
each trial = 30 × 1401
   ↓
reshape = 30 × 3 × 467
   ↓
TT ranks = (15,10)
   ↓
extract TT features
   ↓
feature matrix
   ↓
subject/task labels

In [1]:
pip install pennylane scikit-learn tensorly openpyxl matplotlib pandas

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 3.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 4.7 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 3.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 6.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [pennylane]14 [pennylane]lightning]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

import tensorly as tl
from tensorly.decomposition import tensor_train


# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

import numpy as np


DATA_ROOT = Path("../data/preprocessed")
RESULTS_ROOT = Path("../results/vqc")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASKS = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

TASK_LABELS = {
    "auditory": 0,
    "somatosensory": 1,
    "motor": 2,
    "rest": 3
}


TT_RANKS = [
    1,
    15,
    10,
    1
]


# ============================================================
# FIND FILE
# ============================================================

def find_run_files(subject, task):

    folder = DATA_ROOT / subject / task

    files = sorted(
        folder.glob("*.npz")
    )

    if len(files) == 0:
        raise FileNotFoundError(
            f"No NPZ files found in {folder}"
        )

    return files


# ============================================================
# LOAD DATA
# ============================================================

def load_run(filepath):

    data = np.load(
        filepath,
        allow_pickle=True
    )

    epochs = data["epochs"]

    return epochs

# ============================================================
# RESHAPE ONE TRIAL
# ============================================================

def reshape_trial(trial):

    """
    Input:
        trial = (30, 1401)

    Output:
        tensor = (30, 3, 467)

    because:

        3 * 467 = 1401
    """

    n_channels, n_times = trial.shape

    if n_channels != 30:
        raise ValueError(
            f"Expected 30 channels, "
            f"got {n_channels}"
        )

    if n_times != 1401:
        raise ValueError(
            f"Expected 1401 time points, "
            f"got {n_times}"
        )

    tensor = trial.reshape(
        30,
        3,
        467
    )

    return tensor


# ============================================================
# TT DECOMPOSITION FOR ONE TRIAL
# ============================================================

def tt_decompose_trial(
    trial,
    ranks=TT_RANKS
):

    tensor = reshape_trial(
        trial
    )

    cores = tensor_train(
        tensor,
        rank=ranks
    )

    return cores


# ============================================================
# EXTRACT FEATURES FROM TT CORES
# ============================================================

def extract_tt_features(
    trial,
    ranks=TT_RANKS
):

    cores = tt_decompose_trial(
        trial,
        ranks=ranks
    )

    feature_parts = []

    for core in cores:

        feature_parts.append(
            np.asarray(core)
            .ravel()
        )

    features = np.concatenate(
        feature_parts
    )

    return features

In [6]:
for subject in SUBJECTS:

    for task in TASKS:

        files = find_run_files(
            subject,
            task
        )

        print(
            f"\n{subject} | {task}"
        )

        for filepath in files:

            epochs = load_run(filepath)

            print(
                f"  {filepath.name}: "
                f"{epochs.shape}"
            )


002 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

002 | somatosensory
  run01_epochs.npz: (201, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

002 | motor
  run01_epochs.npz: (73, 30, 1401)
  run02_epochs.npz: (70, 30, 1401)
  run03_epochs.npz: (74, 30, 1401)

002 | rest
  run01_epochs.npz: (1, 30, 1401)

005 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

005 | somatosensory
  run01_epochs.npz: (202, 30, 1401)
  run02_epochs.npz: (198, 30, 1401)

005 | motor
  run01_epochs.npz: (78, 30, 1401)
  run02_epochs.npz: (92, 30, 1401)
  run03_epochs.npz: (76, 30, 1401)

005 | rest
  run01_epochs.npz: (1, 30, 1401)

006 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

006 | somatosensory
  run01_epochs.npz: (203, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

006 | motor
  run01_epochs.npz: (45, 30, 1401)
  run02_epochs.npz: (66, 30, 1401)
  run03_epochs.npz: (55, 30, 1401)

006 | rest

The current code does:
events → MNE Epochs
for every task.

That works for:

Auditory:
stimulus → event → epoch
stimulus → event → epoch
...

and similarly for somatosensory and motor.

But rest doesn't have hundreds of stimulus events.

We have approximately:

5 minutes of resting recording

with the participant simply fixating on a cross.

So your current event detection finds essentially one event, resulting in:

(1, 30, 1401)

That's not a meaningful set of individual rest samples for classification.

Instead, for VQC classification we can divide the continuous rest signal into fixed-length windows.

In [10]:
from pathlib import Path

import numpy as np
import mne


# ============================================================
# PATHS
# ============================================================

LOADED_ROOT = Path("../data/loaded")

SAVE_ROOT = Path("../data/preprocessed_vqc")


files = sorted(
    LOADED_ROOT.rglob("*.npz")
)

print(
    f"Found {len(files)} files"
)


# ============================================================
# VQC EPOCH SETTINGS
# ============================================================

TMIN = -0.2
TMAX = 0.5

REST_WINDOW = 0.7  # seconds


# ============================================================
# PROCESS FILES
# ============================================================

for file in files:

        ## ---------- LOAD DATA ----------

    subject = file.parent.parent.name
    task = file.parent.name
    run = file.stem

    print(subject, task, run)
    
    data = np.load(file, allow_pickle=True)

    signals = data["signals"]
    aux = data["aux"]
    fs = int(data["fs"])
    channel_names = data["channel_names"].tolist()
    positions = data["positions"]
    orientations = data["orientations"]

    print(file.relative_to(LOADED_ROOT))

    print(
        "Signals:",
        signals.shape
    )

    print(
        "Aux:",
        aux.shape
    )

    print(
        "Sampling frequency:",
        fs
    )


    ## ---------- CREATE MNE ----------

    info = mne.create_info(
        ch_names=channel_names,
        sfreq=fs,
        ch_types=["mag"] * len(channel_names)
    )

    raw = mne.io.RawArray(
        signals,
        info
    )

    print(raw)

    ## ---------- FILTERING ----------
     
    raw_filt = raw.copy()
     
    raw_filt.filter(
        l_freq=1.0,
        h_freq=40.0
    )
    
    raw_filt.notch_filter(
        freqs=[60, 120]
    )

    ## ---------- EVENT DETECTION ----------

    task = file.parent.name.lower()
    print(task)
    print(aux.shape)

    if task != "rest":

        if task == "motor":
                trigger = aux[2]
                threshold = 0.5

        else:
            trigger = aux[0]
            threshold = 2.0      
        
        binary = trigger > threshold
    
        onsets = np.where(
            np.diff(binary.astype(int)) == 1
        )[0]
    
        print(
            "Number of events:",
            len(onsets)
        )

        events = np.column_stack(
            [
                onsets,
                np.zeros(
                    len(onsets),
                    dtype=int
                ),
                np.ones(
                    len(onsets),
                    dtype=int
                )
            ]
        )

        print(
            "Events shape:",
            events.shape
        )

        ## ---------- EPOCHING ----------
        
        epochs = mne.Epochs(
            raw_filt,
            events,
            event_id=1,
            tmin=-0.2,
            tmax=0.5,
            baseline=(-0.2, 0),
            preload=True
        )
    
        print(epochs)

        X = epochs.get_data()
        times = epochs.times
        print(
            "Epochs:",
            X.shape
        )


    else:

        # ====================================================
        # RESTING-STATE EPOCHING
        # ====================================================

        print(
            "Creating fixed-length "
            "resting-state windows..."
        )

        # Number of time points per epoch
        N_TIMES = 1401

        # Get filtered continuous data
        rest_data = raw_filt.get_data()

        # Shape: (n_channels, n_samples)

        n_channels, n_samples = rest_data.shape

        # Calculate number of complete windows

        n_windows = (
            n_samples - N_TIMES
        ) // N_TIMES + 1


        print(
            "Number of rest windows:",
            n_windows
        )

        # Extract non-overlapping windows

        X = np.stack(
            [
                rest_data[
                    :,
                    start:start + N_TIMES
                ]
                for start in range(
                    0,
                    n_windows * N_TIMES,
                    N_TIMES
                )
            ]
        )

        # X shape: (n_windows, n_channels, 1401)

        print(
            "Rest windows:",
            X.shape
        )

        times = np.arange(
            N_TIMES
        ) / fs

        # X shape:
        # (n_windows, n_channels, 1401)

    # ========================================================
    # CHECK FINAL SHAPE
    # ========================================================

    print(
        "Final tensor shape:",
        X.shape
    )


    # ========================================================
    # SAVE
    # ========================================================

    SAVE_PATH = (
            SAVE_ROOT /
            subject /
            task /
            f"{run}_epochs.npz"
        )
        
    SAVE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    np.savez_compressed(
        SAVE_PATH,
        epochs=X,
        times=epochs.times,
        fs=fs,
        positions=positions,
        orientations=orientations,
        channel_names=np.array(
            channel_names,
            dtype=object
        )
    )

    print(f"Saved: {SAVE_PATH}")
    print(X.shape)

Found 32 files
002 auditory run01
002/auditory/run01.npz
Signals: (30, 856000)
Aux: (1, 856000)
Sampling frequency: 2000
Creating RawArray with float64 data, n_channels=30, n_times=856000
    Range : 0 ... 855999 =      0.000 ...   428.000 secs
Ready.
<RawArray | 30 x 856000 (428.0 s), ~195.9 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 6601 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing 

Task-related epochs were extracted relative to detected experimental events using a −200 ms to +500 ms window. Since resting-state recordings contain no repeated stimulus events, the continuous resting-state data were instead segmented into non-overlapping 700 ms windows containing 1401 samples, producing fixed-size samples compatible with the task epochs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import tensorly as tl
from tensorly.decomposition import tensor_train


# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

import numpy as np


DATA_ROOT = Path("../data/preprocessed_vqc")
RESULTS_ROOT = Path("../results/vqc")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASKS = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

TASK_LABELS = {
    "auditory": 0,
    "somatosensory": 1,
    "motor": 2,
    "rest": 3
}

In [12]:
for subject in SUBJECTS:

    for task in TASKS:

        files = find_run_files(
            subject,
            task
        )

        print(
            f"\n{subject} | {task}"
        )

        for filepath in files:

            epochs = load_run(filepath)

            print(
                f"  {filepath.name}: "
                f"{epochs.shape}"
            )


002 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

002 | somatosensory
  run01_epochs.npz: (201, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

002 | motor
  run01_epochs.npz: (73, 30, 1401)
  run02_epochs.npz: (70, 30, 1401)
  run03_epochs.npz: (74, 30, 1401)

002 | rest
  run01_epochs.npz: (462, 30, 1401)

005 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

005 | somatosensory
  run01_epochs.npz: (202, 30, 1401)
  run02_epochs.npz: (198, 30, 1401)

005 | motor
  run01_epochs.npz: (78, 30, 1401)
  run02_epochs.npz: (92, 30, 1401)
  run03_epochs.npz: (76, 30, 1401)

005 | rest
  run01_epochs.npz: (438, 30, 1401)

006 | auditory
  run01_epochs.npz: (200, 30, 1401)
  run02_epochs.npz: (200, 30, 1401)

006 | somatosensory
  run01_epochs.npz: (203, 30, 1401)
  run02_epochs.npz: (204, 30, 1401)

006 | motor
  run01_epochs.npz: (45, 30, 1401)
  run02_epochs.npz: (66, 30, 1401)
  run03_epochs.npz: (55, 30, 1401)

006 | 

For each individual epoch/trial, we will do:
X_i ∈ R^ 30×1401

and reshape it to:
X_i ∈ R^30×3×467

because:
3×467=1401.

Then TT decomposition with:
r1 = 15, r2 = 10.

The TT cores are:
G1 ∈ R 30×15
G2 ∈ R 15×3×10
G3 ∈ R 10×467


The resulting representation contains:
30(15)+15(3)(10)+10(467)
= 450 + 450 + 4670 = 5570 parameters.

So ultimately:

5853 samples
      │
      ▼
30 × 1401
      │
      ▼
30 × 3 × 467
      │
      ▼
TT ranks (15,10)
      │
      ▼
5853 × 5570

Trial 1: 30 × 1401 → TT → 5570 features
Trial 2: 30 × 1401 → TT → 5570 features
Trial 3: 30 × 1401 → TT → 5570 features
...
Trial 5853: 30 × 1401 → TT → 5570 features


In [13]:
pip install tensorly scikit-learn pennylane openpyxl

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from pathlib import Path

import numpy as np
import tensorly as tl
from tensorly.decomposition import tensor_train


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path("../data/preprocessed_vqc")

RESULTS_ROOT = Path("../results/vqc")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DATASET
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

TASKS = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

TASK_LABELS = {
    "auditory": 0,
    "somatosensory": 1,
    "motor": 2,
    "rest": 3
}

TT_RANKS = [
    1,
    15,
    10,
    1
]

# ============================================================
# TT FEATURE FUNCTION
# ============================================================

def extract_tt_features(
    trial,
    ranks
):
    """
    Convert one MEG trial into TT parameters.

    Input:
        trial: (30, 1401)

    Reshape:
        (30, 3, 467)

    TT ranks:
        (1, 15, 10, 1)

    Output:
        flattened TT representation
        shape = (5570,)
    """

    # --------------------------------------------------------
    # Check original shape
    # --------------------------------------------------------

    if trial.shape != (30, 1401):

        raise ValueError(
            f"Unexpected trial shape: "
            f"{trial.shape}"
        )


    # --------------------------------------------------------
    # Reshape temporal dimension
    # --------------------------------------------------------

    tensor = trial.reshape(
        30,
        3,
        467
    )


    # --------------------------------------------------------
    # Convert to TensorLy format
    # --------------------------------------------------------

    tensor = tl.tensor(
        tensor,
        dtype=tl.float64
    )

    # --------------------------------------------------------
    # TT decomposition
    # --------------------------------------------------------

    tt_cores = tensor_train(
        tensor,
        rank=ranks
    )

    # --------------------------------------------------------
    # Flatten all TT cores
    # --------------------------------------------------------

    features = np.concatenate(
        [
            tl.to_numpy(core).ravel()
            for core in tt_cores
        ]
    )

    return features

# ============================================================
# PROCESS ALL DATA
# ============================================================

X_tt = []

y = []

subjects = []

tasks = []

runs = []

trial_numbers = []


for subject in SUBJECTS:

    for task in TASKS:

        task_folder = (
            DATA_ROOT /
            subject /
            task
        )


        files = sorted(
            task_folder.glob(
                "*_epochs.npz"
            )
        )


        if not files:

            print(
                f"WARNING: no files found: "
                f"{task_folder}"
            )

            continue


        print(
            f"\n{subject} | "
            f"{task} | "
            f"{len(files)} runs"
        )


        label = TASK_LABELS[task]


        for file in files:

            print(
                f"  Processing "
                f"{file.name}"
            )


            data = np.load(
                file,
                allow_pickle=True
            )


            epochs = data["epochs"]


            print(
                f"    Epochs: "
                f"{epochs.shape}"
            )


            # ------------------------------------------------
            # Process every trial separately
            # ------------------------------------------------

            for trial_idx, trial in enumerate(
                epochs
            ):


                features = extract_tt_features(
                    trial,
                    TT_RANKS
                )


                X_tt.append(
                    features
                )


                y.append(
                    label
                )


                subjects.append(
                    subject
                )


                tasks.append(
                    task
                )


                runs.append(
                    file.stem
                )


                trial_numbers.append(
                    trial_idx
                )


# ============================================================
# CONVERT TO NUMPY
# ============================================================

X_tt = np.asarray(
    X_tt,
    dtype=np.float64
)

y = np.asarray(
    y,
    dtype=np.int64
)

subjects = np.asarray(
    subjects
)

tasks = np.asarray(
    tasks
)

runs = np.asarray(
    runs
)

trial_numbers = np.asarray(
    trial_numbers
)


# ============================================================
# CHECK
# ============================================================

print("\n")
print("=" * 70)
print("TT FEATURE DATASET")
print("=" * 70)

print(
    "X_tt shape:",
    X_tt.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "Unique subjects:",
    np.unique(subjects)
)

print(
    "Unique tasks:",
    np.unique(tasks)
)

print(
    "TT ranks:",
    TT_RANKS
)


# ============================================================
# EXPECTED FEATURE COUNT
# ============================================================

expected_features = (
    30 * 15
    + 15 * 3 * 10
    + 10 * 467
)

print(
    "Expected TT parameters:",
    expected_features
)


if X_tt.shape[1] != expected_features:

    raise ValueError(
        "Unexpected TT feature dimension!"
    )


# ============================================================
# SAVE
# ============================================================

save_path = (
    RESULTS_ROOT /
    "tt_features_r15_r10.npz"
)


np.savez_compressed(

    save_path,

    X_tt=X_tt,

    y=y,

    subjects=subjects,

    tasks=tasks,

    runs=runs,

    trial_numbers=trial_numbers,

    tt_ranks=np.asarray(
        [15, 10]
    )

)


print(
    "\nSaved:"
)

print(
    save_path
)


002 | auditory | 2 runs
  Processing run01_epochs.npz
    Epochs: (200, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (200, 30, 1401)

002 | somatosensory | 2 runs
  Processing run01_epochs.npz
    Epochs: (201, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (204, 30, 1401)

002 | motor | 3 runs
  Processing run01_epochs.npz
    Epochs: (73, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (70, 30, 1401)
  Processing run03_epochs.npz
    Epochs: (74, 30, 1401)

002 | rest | 1 runs
  Processing run01_epochs.npz
    Epochs: (462, 30, 1401)

005 | auditory | 2 runs
  Processing run01_epochs.npz
    Epochs: (200, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (200, 30, 1401)

005 | somatosensory | 2 runs
  Processing run01_epochs.npz
    Epochs: (202, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (198, 30, 1401)

005 | motor | 3 runs
  Processing run01_epochs.npz
    Epochs: (78, 30, 1401)
  Processing run02_epochs.npz
    Epochs: (92, 30, 1401)
  Processing run03

In [2]:
from pathlib import Path

import numpy as np
import tensorly as tl
from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    "../data/preprocessed_vqc"
)

TT_PATH = Path(
    "../results/vqc/tt_features_r15_r10.npz"
)


# ============================================================
# SETTINGS
# ============================================================

SUBJECT = "002"
TASK = "auditory"
RUN = "run01_epochs.npz"

N_TEST_TRIALS = 10

TT_RANKS = [
    1,
    15,
    10,
    1
]


# ============================================================
# LOAD ORIGINAL DATA
# ============================================================

original_path = (
    DATA_ROOT /
    SUBJECT /
    TASK /
    RUN
)

data = np.load(
    original_path,
    allow_pickle=True
)

epochs = data["epochs"]


print("=" * 70)
print("TT RECONSTRUCTION TEST")
print("=" * 70)

print(
    "Original dataset:",
    epochs.shape
)

print(
    "Testing:",
    SUBJECT,
    TASK,
    RUN
)


# ============================================================
# SELECT TRIALS
# ============================================================

n_trials = min(
    N_TEST_TRIALS,
    len(epochs)
)

rng = np.random.default_rng(
    42
)

trial_indices = rng.choice(
    len(epochs),
    size=n_trials,
    replace=False
)

print(
    "Trial indices:",
    trial_indices
)


# ============================================================
# TEST EACH TRIAL
# ============================================================

errors = []

correlations = []


for trial_idx in trial_indices:

    # --------------------------------------------------------
    # Original trial
    # --------------------------------------------------------

    original = epochs[
        trial_idx
    ]

    print(
        f"\nTrial {trial_idx}"
    )

    print(
        "Original shape:",
        original.shape
    )


    # --------------------------------------------------------
    # Reshape
    # --------------------------------------------------------

    tensor = original.reshape(
        30,
        3,
        467
    )


    # --------------------------------------------------------
    # TT decomposition
    # --------------------------------------------------------

    tensor_tl = tl.tensor(
        tensor,
        dtype=tl.float64
    )

    cores = tensor_train(
        tensor_tl,
        rank=TT_RANKS
    )


    # --------------------------------------------------------
    # Reconstruct
    # --------------------------------------------------------

    reconstructed = tt_to_tensor(
        cores
    )

    reconstructed = tl.to_numpy(
        reconstructed
    )


    # --------------------------------------------------------
    # Reshape back
    # --------------------------------------------------------

    reconstructed = (
        reconstructed
        .reshape(30, 1401)
    )


    # --------------------------------------------------------
    # Reconstruction error
    # --------------------------------------------------------

    numerator = np.linalg.norm(
        original - reconstructed
    )

    denominator = np.linalg.norm(
        original
    )

    relative_error = (
        numerator / denominator
    )


    # --------------------------------------------------------
    # Correlation
    # --------------------------------------------------------

    correlation = np.corrcoef(
        original.ravel(),
        reconstructed.ravel()
    )[0, 1]


    errors.append(
        relative_error
    )

    correlations.append(
        correlation
    )


    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        "Reconstructed shape:",
        reconstructed.shape
    )

    print(
        "Relative reconstruction error:",
        f"{relative_error:.6f}"
    )

    print(
        "Correlation:",
        f"{correlation:.6f}"
    )


# ============================================================
# SUMMARY
# ============================================================

errors = np.asarray(
    errors
)

correlations = np.asarray(
    correlations
)


print("\n")
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Mean reconstruction error:",
    f"{errors.mean():.6f}"
)

print(
    "Std reconstruction error:",
    f"{errors.std():.6f}"
)

print(
    "Mean correlation:",
    f"{correlations.mean():.6f}"
)

print(
    "Minimum correlation:",
    f"{correlations.min():.6f}"
)

print(
    "Maximum correlation:",
    f"{correlations.max():.6f}"
)


# ============================================================
# PASS / FAIL
# ============================================================

if np.all(np.isfinite(errors)):

    print(
        "\nPASS: reconstruction values "
        "are finite."
    )

else:

    print(
        "\nFAIL: NaN or infinite "
        "reconstruction error."
    )


if np.all(np.isfinite(correlations)):

    print(
        "PASS: correlations are finite."
    )

else:

    print(
        "FAIL: NaN or infinite "
        "correlations."
    )

TT RECONSTRUCTION TEST
Original dataset: (200, 30, 1401)
Testing: 002 auditory run01_epochs.npz
Trial indices: [ 16 148  17 126  85  84 138  18  40 168]

Trial 16
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.153426
Correlation: 0.988100

Trial 148
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.103286
Correlation: 0.994329

Trial 17
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.107583
Correlation: 0.994181

Trial 126
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.086209
Correlation: 0.995653

Trial 85
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.065222
Correlation: 0.997635

Trial 84
Original shape: (30, 1401)
Reconstructed shape: (30, 1401)
Relative reconstruction error: 0.105911
Correlation: 0.994290

Trial 138
Original shape: (30, 1401)
Reconstructed shape: 

TT decomposition with ranks (15,10) provides a substantially compressed representation while preserving the overall structure of the MEG trials, with a mean relative reconstruction error of approximately 10.5% and mean correlation of approximately 0.994 in the tested trials.

In [3]:
from pathlib import Path

import numpy as np


# ============================================================
# LOAD
# ============================================================

PATH = Path(
    "../results/vqc/tt_features_r15_r10.npz"
)

data = np.load(
    PATH,
    allow_pickle=True
)


X = data["X_tt"]
y = data["y"]
subjects = data["subjects"]
tasks = data["tasks"]
runs = data["runs"]
trial_numbers = data["trial_numbers"]


# ============================================================
# BASIC TESTS
# ============================================================

print("=" * 70)
print("TT FEATURE VALIDATION")
print("=" * 70)


print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "Number of subjects:",
    len(np.unique(subjects))
)

print(
    "Subjects:",
    np.unique(subjects)
)

print(
    "Tasks:",
    np.unique(tasks)
)


# ============================================================
# SHAPE
# ============================================================

assert X.ndim == 2

assert X.shape[1] == 5570

assert len(X) == len(y)

assert len(X) == len(subjects)

assert len(X) == len(tasks)

assert len(X) == len(runs)

assert len(X) == len(trial_numbers)


print(
    "\nPASS: dimensions are correct."
)


# ============================================================
# NaN / INF
# ============================================================

print(
    "\nNaN values:",
    np.isnan(X).sum()
)

print(
    "Infinite values:",
    np.isinf(X).sum()
)


assert not np.isnan(X).any()

assert not np.isinf(X).any()


print(
    "PASS: no NaN or infinite values."
)


# ============================================================
# FEATURE STATISTICS
# ============================================================

print("\n")
print("=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

print(
    "Minimum:",
    X.min()
)

print(
    "Maximum:",
    X.max()
)

print(
    "Mean:",
    X.mean()
)

print(
    "Std:",
    X.std()
)


# ============================================================
# LABEL COUNTS
# ============================================================

print("\n")
print("=" * 70)
print("TASK COUNTS")
print("=" * 70)

for task in np.unique(tasks):

    count = np.sum(
        tasks == task
    )

    print(
        f"{task:15s}: {count}"
    )


# ============================================================
# SUBJECT COUNTS
# ============================================================

print("\n")
print("=" * 70)
print("SUBJECT COUNTS")
print("=" * 70)

for subject in np.unique(subjects):

    count = np.sum(
        subjects == subject
    )

    print(
        f"{subject}: {count}"
    )


print("\n")
print("=" * 70)
print("ALL TESTS PASSED")
print("=" * 70)

TT FEATURE VALIDATION
X shape: (5853, 5570)
y shape: (5853,)
Number of subjects: 4
Subjects: ['002' '005' '006' '093']
Tasks: ['auditory' 'motor' 'rest' 'somatosensory']

PASS: dimensions are correct.

NaN values: 0
Infinite values: 0
PASS: no NaN or infinite values.


FEATURE STATISTICS
Minimum: -0.6932675315876409
Maximum: 0.9963358569799472
Mean: 0.0023520568911343915
Std: 0.06695370168626955


TASK COUNTS
auditory       : 1600
motor          : 850
rest           : 1774
somatosensory  : 1629


SUBJECT COUNTS
002: 1484
005: 1484
006: 1411
093: 1474


ALL TESTS PASSED


TRAIN:
002
005
006

TEST:
093

LOSO: Leave-One-Subject-Out cross-validation (LOSO)

Instead of having one fixed test subject, we rotate which subject is the test subject.

In [ ]:
from tensorly.tt_tensor import tt_to_tensor

In [1]:
from pathlib import Path

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# PATHS
# ============================================================

DATA_PATH = (
    Path("../results/vqc")
    / "tt_features_r15_r10.npz"
)

RESULTS_ROOT = Path(
    "../results/vqc"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD TT DATA
# ============================================================

data = np.load(
    DATA_PATH,
    allow_pickle=True
)

X_tt = data["X_tt"]

y = data["y"]

subjects = data["subjects"]

tasks = data["tasks"]


print(
    "TT features:",
    X_tt.shape
)


# ============================================================
# QUANTUM DIMENSIONS TO TEST
# ============================================================

N_COMPONENTS = [
    4,
    8,
    12,
    16
]


SUBJECT_LIST = np.unique(
    subjects
)


# ============================================================
# LEAVE-ONE-SUBJECT-OUT
# ============================================================

for test_subject in SUBJECT_LIST:

    print("\n")
    print("=" * 70)

    print(
        "TEST SUBJECT:",
        test_subject
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Train/test masks
    # --------------------------------------------------------

    train_mask = (
        subjects != test_subject
    )

    test_mask = (
        subjects == test_subject
    )


    X_train = X_tt[
        train_mask
    ]

    X_test = X_tt[
        test_mask
    ]


    print(
        "Training:",
        X_train.shape
    )

    print(
        "Testing:",
        X_test.shape
    )


    # ========================================================
    # STANDARDISATION
    # ========================================================

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train
        )
    )

    X_test_scaled = (
        scaler.transform(
            X_test
        )
    )


    # ========================================================
    # PCA
    # ========================================================

    for n_components in N_COMPONENTS:

        pca = PCA(
            n_components=n_components
        )


        X_train_pca = (
            pca.fit_transform(
                X_train_scaled
            )
        )


        X_test_pca = (
            pca.transform(
                X_test_scaled
            )
        )


        variance = (
            np.sum(
                pca.explained_variance_ratio_
            )
        )


        print(
            f"{n_components:2d} components | "
            f"variance retained = "
            f"{variance:.4f} | "
            f"train = {X_train_pca.shape} | "
            f"test = {X_test_pca.shape}"
        )

TT features: (5853, 5570)


TEST SUBJECT: 002
Training: (4369, 5570)
Testing: (1484, 5570)
 4 components | variance retained = 0.2173 | train = (4369, 4) | test = (1484, 4)
 8 components | variance retained = 0.3481 | train = (4369, 8) | test = (1484, 8)
12 components | variance retained = 0.4379 | train = (4369, 12) | test = (1484, 12)
16 components | variance retained = 0.5091 | train = (4369, 16) | test = (1484, 16)


TEST SUBJECT: 005
Training: (4369, 5570)
Testing: (1484, 5570)
 4 components | variance retained = 0.2153 | train = (4369, 4) | test = (1484, 4)
 8 components | variance retained = 0.3450 | train = (4369, 8) | test = (1484, 8)
12 components | variance retained = 0.4289 | train = (4369, 12) | test = (1484, 12)
16 components | variance retained = 0.4961 | train = (4369, 16) | test = (1484, 16)


TEST SUBJECT: 006
Training: (4442, 5570)
Testing: (1411, 5570)
 4 components | variance retained = 0.2135 | train = (4442, 4) | test = (1411, 4)
 8 components | variance retained